In [2]:
import json
import glob
import os

data_dir = r"C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\src\data_lake\JSON"

# Archivos por tipo
athletes_files = glob.glob(os.path.join(data_dir, "*.athletes.json"))
issues_files = glob.glob(os.path.join(data_dir, "*.issues.json"))
live_files = glob.glob(os.path.join(data_dir, "*.live.json"))
race_files = glob.glob(os.path.join(data_dir, "*.race.json"))

# Diccionarios para guardar los datos
all_athletes = {}
all_issues = {}
all_live = {}
all_race = {}

def get_race_id(filepath):
    filename = os.path.basename(filepath)
    return filename.split(".")[0]

# Cargar archivos athletes
for a_file in athletes_files:
    race_id = get_race_id(a_file)
    with open(a_file, "r", encoding="utf-8") as f:
        all_athletes[race_id] = json.load(f)

# Cargar archivos issues
for i_file in issues_files:
    race_id = get_race_id(i_file)
    with open(i_file, "r", encoding="utf-8") as f:
        all_issues[race_id] = json.load(f)

# Cargar archivos live
for l_file in live_files:
    race_id = get_race_id(l_file)
    with open(l_file, "r", encoding="utf-8") as f:
        all_live[race_id] = json.load(f)

# Cargar archivos race
for r_file in race_files:
    race_id = get_race_id(r_file)
    with open(r_file, "r", encoding="utf-8") as f:
        all_race[race_id] = json.load(f)

print("Athletes cargados:", list(all_athletes.keys()))
print("Issues cargados:", list(all_issues.keys()))
print("Live cargados:", list(all_live.keys()))
print("Race cargados:", list(all_race.keys()))

Athletes cargados: ['edreams-mitja-marato-barcelona-2025-by-brooks', 'skoda-titan-desert-morocco-2025', 'xxiv-movistar-madrid-medio-maraton-2025', 'zurich-marato-barcelona-2025']
Issues cargados: ['edreams-mitja-marato-barcelona-2025-by-brooks', 'skoda-titan-desert-morocco-2025', 'xxiv-movistar-madrid-medio-maraton-2025', 'zurich-marato-barcelona-2025']
Live cargados: ['edreams-mitja-marat-barcelona-2022']
Race cargados: ['edreams-mitja-marat-barcelona-2022']


In [1]:
import os
import glob
import json

# ==============================
# CONFIGURACIÓN
# ==============================

data_dir = r"C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\src\data_lake\JSON\maratones4"
output_dir = os.path.join(data_dir, "estructuras")
os.makedirs(output_dir, exist_ok=True)

# MODO DE EJECUCIÓN
# True  → procesa SOLO una carrera
# False → procesa TODAS las carreras
SINGLE_RACE_MODE = False

# Solo se usa si SINGLE_RACE_MODE = True
selected_race_id = "zurich-maraton-de-sevilla-2022"

# Tipos de archivos a procesar
file_types = ["athletes", "issues", "live", "race"]

# ==============================
# FUNCIONES
# ==============================

def merge_structures(base, new, max_unique=0):
    if isinstance(base, dict) and isinstance(new, dict):
        merged = dict(base)
        for key, val in new.items():
            if key in merged:
                merged[key] = merge_structures(merged[key], val, max_unique)
            else:
                merged[key] = val
        return merged
    elif isinstance(base, set) and isinstance(new, set):
        merged_set = base | new
        if len(merged_set) > max_unique:
            return "*"
        return merged_set
    elif isinstance(base, set):
        if isinstance(new, (dict, list)):
            return "*|dict_or_list"
        return merge_structures(base, {new}, max_unique)
    elif isinstance(new, set):
        if isinstance(base, (dict, list)):
            return "*|dict_or_list"
        return merge_structures({base}, new, max_unique)
    else:
        if isinstance(base, (dict, list)) or isinstance(new, (dict, list)):
            return "*|dict_or_list"
        return {base, new}

def extract_structure_unique(data):
    if isinstance(data, dict):
        return {k: extract_structure_unique(v) for k, v in data.items()}
    elif isinstance(data, list):
        if not data:
            return {"[]": {}}
        struct = extract_structure_unique(data[0])
        for item in data[1:]:
            struct = merge_structures(struct, extract_structure_unique(item))
        return {"[]": struct}
    else:
        return {data}

def sets_to_lists(obj):
    if isinstance(obj, dict):
        return {k: sets_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, set):
        return [sets_to_lists(v) for v in obj]
    elif isinstance(obj, list):
        return [sets_to_lists(v) for v in obj]
    else:
        return obj

def get_race_id(filepath):
    return os.path.basename(filepath).split(".")[0]

# ==============================
# PROCESAMIENTO
# ==============================

print("\n==============================")
print(" INICIANDO PROCESAMIENTO")
print("==============================\n")

for f_type in file_types:
    files = glob.glob(os.path.join(data_dir, f"*.{f_type}.json"))

    if not files:
        print(f"No se encontraron archivos para tipo: {f_type}")
        continue

    for f_path in files:
        race_id = get_race_id(f_path)

        if SINGLE_RACE_MODE and race_id != selected_race_id:
            continue

        print(f"Procesando {f_type.upper()} → {race_id}")

        with open(f_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        structure = extract_structure_unique(data)
        structure_serializable = sets_to_lists(structure)

        output_path = os.path.join(output_dir, f"{race_id}_structure_{f_type}.json")

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(structure_serializable, f, indent=4, ensure_ascii=False)

        print(f"   Guardado: {output_path}")

print("\n==============================")
print(" PROCESAMIENTO FINALIZADO")
print("==============================")



 INICIANDO PROCESAMIENTO

Procesando ATHLETES → 44-zurich-maraton-san-sebastian-2022
   Guardado: C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\src\data_lake\JSON\maratones4\estructuras\44-zurich-maraton-san-sebastian-2022_structure_athletes.json
Procesando ATHLETES → 45-zurich-maraton-san-sebastian-2023


KeyboardInterrupt: 

In [9]:
import json
import os

data_dir = r"C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\src\data_lake\JSON\maratones4"
selected_race_id = "zurich-marato-barcelona-2022.athletes"

# Ruta completa al archivo JSON
json_path = os.path.join(data_dir, f"{selected_race_id}.json")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Claves del primer elemento del JSON:")
print(data[0].keys())

Claves del primer elemento del JSON:
dict_keys(['dorsal', 'id', 'wave', 'event', 'name', 'surname', 'gender', 'birthdate', 'CÓDIGO POSTAL', 'PROVINCIA', 'nationality', 'CodigoInternacional', 'club', 'Color dorsal', 'TIEMPO ESTIMADO', 'Federado', 'category', 'chip', 'status', 'importId', 'team', 'team_type', 'featured', 'user_id', 'attributes', 'realStatus', 'startTime', 'startRawTime', 'startNetTime', 'distance', 'leader_weight', 'maxConsecutiveSplitsMissing', 'splitsMissing', 'issuesCount', 'splitsSeen', 'last_split_seen', 'times', 'rankings', 'leader', 'calculatedTimes', 'commentatorTimes', 'predictive', 'backups', 'locations', 'mst', 'geo', 'penalties', 'color'])


In [ ]:
## CAMBIARLO PARA QUE SE UED HACER POR CARRERA TAMBIEN

import json
import os
import glob

ATHLETE_ID = 'L6554352'  # cambia por el ID del atleta que quieras

# obtener archivos
athletes_files = glob.glob(os.path.join(data_dir, "*.athletes.json"))
issues_files = glob.glob(os.path.join(data_dir, "*.issues.json"))

if not athletes_files:
    raise FileNotFoundError("No se encontró ningún archivo de atletas en el directorio.")
if not issues_files:
    raise FileNotFoundError("No se encontró ningún archivo de issues en el directorio.")

# carga de datos
for fpath in athletes_files:
    with open(fpath, "r", encoding="utf-8") as f:
        athletes_data.extend(json.load(f))  # agregamos todos los atletas de cada archivo

issues_data = []
for fpath in issues_files:
    with open(fpath, "r", encoding="utf-8") as f:
        issues_data.extend(json.load(f))  # agregamos todas las listas de issues

#buscarlo
athlete = next((a for a in athletes_data if str(a.get("id")) == str(ATHLETE_ID)), None)
if athlete is None:
    raise ValueError(f"No se encontró ningún atleta con id {ATHLETE_ID}")

#busar issues
for issue_list in issues_data:
    # cada issue_list puede ser una lista de issues
    if isinstance(issue_list, list):
        for issue in issue_list:
            if str(issue.get("athlete_id")) == str(ATHLETE_ID) or issue.get("athlete_id") == athlete.get("id"):
                athlete_issues.append(issue)
    elif isinstance(issue_list, dict):
        # si algún archivo tiene un dict en lugar de lista
        if str(issue_list.get("athlete_id")) == str(ATHLETE_ID):
            athlete_issues.append(issue_list)

output_athlete = os.path.join(data_dir, f"athlete_{ATHLETE_ID}.json")
output_issues = os.path.join(data_dir, f"athlete_{ATHLETE_ID}_issues.json")

with open(output_athlete, "w", encoding="utf-8") as f:
    json.dump(athlete, f, ensure_ascii=False, indent=4)

with open(output_issues, "w", encoding="utf-8") as f:
    json.dump(athlete_issues, f, ensure_ascii=False, indent=4)

print(f"Archivos guardados:\n - {output_athlete}\n - {output_issues}")

In [ ]:
athletes_time_df = dfs['athletes_time_df']

import pandas as pd
import numpy as np

numeric_pairs = [
    ("netTime", "time"),
    ("originalTime", "raw_times_official"),
    ("originalTime", "raw_times_real"),
    ("raw_backupOffset", "offset"),
    ("raw_times_official", "raw_times_real"),
    ("raw_times_rawTime", "raw_rawTime"),
]

for col1, col2 in numeric_pairs:
    df_pair = athletes_time_df[[col1, col2]].dropna()
    exact_match = (df_pair[col1] == df_pair[col2]).all()
    diff_mean = (df_pair[col1] - df_pair[col2]).abs().mean()
    print(f"{col1} vs {col2}: Exact match? {exact_match}, Mean absolute difference: {diff_mean}")

datetime_pairs = [
    ("rawTime", "raw_rawTime"),
    ("raw_originalTime", "raw_times_rawTime"),
]

for col1, col2 in datetime_pairs:
    df_pair = athletes_time_df[[col1, col2]].dropna()
    exact_match = (df_pair[col1] == df_pair[col2]).all()
    diff_mean = (df_pair[col1] - df_pair[col2]).abs().mean()
    print(f"{col1} vs {col2}: Exact match? {exact_match}, Mean absolute difference: {diff_mean}")

In [ ]:
import os

output_folder = r"C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\raiz\datos\historicos\calcular_peso"
os.makedirs(output_folder, exist_ok=True)

for name, df in dfs_final.items():
    print(f"Guardando {name}...")

    parquet_path = os.path.join(output_folder, f"{name}.parquet")
    df.to_parquet(parquet_path, index=False)
    
    csv_path = os.path.join(output_folder, f"{name}.csv.gz")
    df.to_csv(csv_path, index=False, compression='gzip')

print("\n✅ Todos los DataFrames se han guardado correctamente en:")
print(output_folder)

In [4]:
data_dir = r"C:\Users\mario\Desktop\MasterCienciadeDatos\TFM\TFM_MarioSoto\src\data_lake\JSON\maratones2"

import json
import os
from datetime import datetime, timezone

def get_athlete_split_time(json_file_path, athlete_id, event_name, split_name):
    with open(json_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Si la raíz es un dict con "[]", convertimos a lista
    if isinstance(data, dict) and "[]" in data:
        athletes = data["[]"]
    elif isinstance(data, list):
        athletes = data
    else:
        raise ValueError("Formato de JSON inesperado en la raíz")

    for athlete in athletes:
        if str(athlete.get("id")) != str(athlete_id):
            continue

        # Eventos
        events_container = athlete.get("events", {})
        if isinstance(events_container, dict) and "[]" in events_container:
            events = events_container["[]"]
        elif isinstance(events_container, list):
            events = events_container
        else:
            events = []

        for event in events:
            if event.get("event") != event_name:
                continue

            times = event.get("times", {})
            split = times.get(split_name, {})
            raw_info = split.get("raw", None)
            if raw_info:
                return raw_info

    return None


filename = "xl-zurich-maraton-de-sevilla-2025.athletes.json"

# Ejemplo de uso
json_path = os.path.join(data_dir, filename)
athlete_id = "12213"
event_name = "Maratón"
split_name = "30K"

raw_time = get_athlete_split_time(json_path, athlete_id, event_name, split_name)

timestamp_ms = raw_time.get('rawTime')

timestamp_s = timestamp_ms / 1000

dt = datetime.fromtimestamp(timestamp_s, tz=timezone.utc)

#print(dt)  

formatted_date = dt.strftime("%Y-%m-%d %H:%M:%S.%f")
print(formatted_date)  

2025-02-23 10:25:39.300000


In [3]:
from PIL import Image
import os

# Lista de imágenes (en el orden que quieras)
imagenes = [
    r"C:\Users\mario\Downloads\gen_decil_1.png",
    r"C:\Users\mario\Downloads\gen_decil_2.png",
    r"C:\Users\mario\Downloads\gen_decil_4.png",
    r"C:\Users\mario\Downloads\gen_decil_8.png"
]

# Cargar imágenes
imgs = [Image.open(p) for p in imagenes]

# Tamaño de la imagen final
total_width = sum(img.width for img in imgs)
max_height = max(img.height for img in imgs)

# Crear imagen final
combined = Image.new("RGB", (total_width, max_height), color=(255,255,255))

# Pegar una al lado de otra
x_offset = 0
for img in imgs:
    combined.paste(img, (x_offset, 0))
    x_offset += img.width

# Guardar
output = r"C:\Users\mario\Downloads\gen_decil.png"
combined.save(output)

print("Imagen creada:", output)

Imagen creada: C:\Users\mario\Downloads\gen_decil.png


In [ ]:
from PIL import Image

# Lista de imágenes (en el orden que quieras)
imagenes = [
    r"C:\Users\mario\Downloads\curvas_acum_regr_1.png",
    r"C:\Users\mario\Downloads\scatter_regr.png"
]

# Cargar imágenes
imgs = [Image.open(p) for p in imagenes]

# Tamaño de la imagen final
max_width = max(img.width for img in imgs)
total_height = sum(img.height for img in imgs)

# Crear imagen final
combined = Image.new("RGB", (max_width, total_height), color=(255, 255, 255))

# Pegar una debajo de otra
y_offset = 0
for img in imgs:
    combined.paste(img, (0, y_offset))
    y_offset += img.height

# Guardar
output = r"C:\Users\mario\Downloads\scatter.png"
combined.save(output)

print("Imagen creada:", output)

Imagen creada: C:\Users\mario\Downloads\scatter.png
